In [2]:
import os
import tensorflow as tf
import numpy as np
import math
from random import sample,shuffle
from PIL import Image
import matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense,Conv2D,Flatten,Reshape,Conv2DTranspose,BatchNormalization,Conv1D,Input,Layer
from tensorflow.keras import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
import seaborn as sns
import keras


In [3]:
!pip install librosa
!pip install spotipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 409.8/409.8 kB 9.2 MB/s eta 0:00:00:00:01


In [4]:
import numpy as np
from random import sample,shuffle
import os
import tensorflow as tf
import numpy.ma as ma
import requests
import librosa
from skimage.transform import resize
from tqdm import tqdm

In [5]:
 def download_preview_with_url(track_url, track_id):
        preview = requests.get(track_url)
        filename = f'/kaggle/working/{track_id}.mp3'
        with open(filename, 'wb') as f:
            f.write(preview.content)
        composite_image = convert_audio_to_composite_image(filename)
        os.remove(filename)
        return composite_image
def convert_audio_to_composite_image(filepath_to_audio, image_size=(128,512), n_mels=128, fmax=8000,):
    
    signal, sr = librosa.load(filepath_to_audio)
    
    mels = librosa.power_to_db(librosa.feature.melspectrogram(y=signal, sr=sr, n_mels=n_mels, fmax=fmax), ref=np.max)
    mel_image = (((80+mels)/80)*255)
    mel_image = np.flip(mel_image, axis=0)
    mel_image = resize(mel_image, (128,512)).astype(np.uint8)
    
    mfcc = librosa.power_to_db(librosa.feature.mfcc(y=signal, sr=sr, n_mfcc=128, fmax=8000), ref=np.max)
    mfcc_image = (((80+mfcc)/80)*255)
    mfcc_image = np.flip(mfcc_image, axis=0)
    mfcc_image = resize(mfcc_image, (128,512)).astype(np.uint8)
    
    chromagram = librosa.feature.chroma_cqt(y=signal, sr=sr)
    chroma_image = resize(chromagram*255, (128,512)).astype(np.uint8)
    
    composite = np.dstack((mel_image, mfcc_image, chroma_image))

    return composite

In [6]:
class AudioDataGenerator(tf.keras.utils.Sequence):
    def __init__(self,directory,
                image_size,color_mode='grayscale',
                batch_size=32,shuffle=False,
                sample_size=None,train_test_split=False,
                test_size=.2,
                file_list=None,
                name="Generator",
                output_channel_index=None,num_output_channels=1,output_size=None,
                threshold_level=0,):
        self.dir=directory
        self._image_size=image_size
        self._img_height=image_size[0]
        self._img_width=image_size[1]
        self._color_mode=color_mode
        if color_mode=='grayscale':
            self._img_channels=1
        elif color_mode=='rgb':
            self._img_channels=3
        self.output_channel_index=output_channel_index
        self.num_output_channels=num_output_channels
        self.output_size=output_size
        self.shuffle=shuffle
        self.sample_size=sample_size
        self.test_size=test_size
        self.batch_size=batch_size
        self.threshold_level=threshold_level
        if file_list==None:
            self._files=self.__get_images_from_directory(directory)
        else:
            try:
                self._files=self.__collect_image_files(file_list)
            except TypeError:
                print('file_list is not a list')
        if train_test_split:
            self.__train_test_split(self._files)
        else:
            print(f'Found {len(self._files)} files for {name} set')
        self.size=len(self._files)
        self.on_epoch_end()
    def __len__(self):
      return len(self._files)//self.batch_size
    def __getitem__(self,index=0,return_filename=False,get_all_tiles=False,num_tiles=4,image=False,image_data=None,filename=None):
        if filename==None:
            batch=self._files[index*self.batch_size:index*self.batch_size+self.batch_size]
        else:
            batch=[filename]
        X,y=self.__get_data(batch,image,image_data)
        if self.output_channel_index !=None:
            X=X[:,:,:,self.output_channel_index:self.output_channel_index+self.num_output_channels]
            y=y[:,:,:,self.output_channel_index:self.output_channel_index+self.num_output_channels]
        if self.output_size !=None:
            if get_all_tiles==False:
                if self.output_size[1]<self._img_width:
                    rand_x_index=np.random.randint(low=0,high=self._img_width-self.output_size[1])
                else:
                    rand_x_index=0
                if self.output_size[0]<self._img_height:
                    rand_y_index=np.random.randint(low=0,high=self._img_height-self.output_size[0])
                else:
                    rand_y_index=0
                X=X[:,rand_y_index:rand_y_index+self.output_size[0],rand_x_index:rand_x_index+self.output_size[1],:]
                y=X
            else:
                if num_tiles >1:
                    slice_size=(self._img_width-self.output_size[1])//(num_tiles-1)
                else:
                    slice_size=0
                all_tiles=[]
                new_batch=[]
                for idx,img in enumerate(X):
                    for i in range(num_tiles):
                        all_tiles.append(img[:,i*slice_size:(i*slice_size)+self.output_size[1],:])
                        new_batch.append(batch[idx])
                X=np.array(all_tiles)
                y=X
                if return_filename:
                    batch=new_batch
        if return_filename:
            return batch,X,y
        else:
            return X,y
    def on_epoch_end(self):
        if self.shuffle:
            shuffle(self._files)
    def get_vector_from_preview_link(self, track_url, track_id, num_tiles=32):
        img = download_preview_with_url(track_url, track_id)
        return self.__getitem__(get_all_tiles=True, num_tiles=num_tiles, image=True, image_data=img, filename=track_id)
    def __get_data(self,batch,image=False,image_data=None):
        X=np.empty((self.batch_size,self._img_height,self._img_width,self._img_channels))
        for i,file in enumerate(batch):
            if image==False:
                path=self.dir+file
                img=tf.keras.preprocessing.image.load_img(path,color_mode=self._color_mode)
            else:
                img=image_data
            scale=1./255
            img=scale*np.array(img)
            if self.threshold_level>0:
                img_min=img.min()
                img_max=img.max()
                threshold=self.threshold_level*(img_max-img_min)+img_min
                mx=ma.masked_where(img<threshold,img,copy=True)
                img=(mx.filled(fill_value=threshold)-threshold)
                img=img*(img_max/img.max())
            X[i,]=tf.convert_to_tensor(img)
        y=X
        return X,y
    def take(self,index=1,return_filename=False,get_all_tiles=False,num_tiles=4):
        return self.__getitem__(index,return_filename,get_all_tiles,num_tiles)
    def __train_test_split(self,files):
        if self.shuffle:
            shuffle(files)
        file_list_length=len(files)
        print(f"Total {len(files)} images")
        test_split=int(file_list_length*(1-self.test_size))
        train_files=files[:test_split]
        test_files=files[test_split:]
        self.train=AudioDataGenerator(self.dir,self._image_size,batch_size=self.batch_size,color_mode=self._color_mode,shuffle=self.shuffle,file_list=train_files,name='Train',output_channel_index=self.output_channel_index,output_size=self.output_size)
        self.test = AudioDataGenerator(self.dir, self._image_size, batch_size=self.batch_size, color_mode=self._color_mode, shuffle=self.shuffle, file_list=test_files, name='Test', output_channel_index=self.output_channel_index, output_size=self.output_size)
    def __collect_image_files(self,files):
        filetypes=['png','jpg','jpeg','webp']
        return [file for file in files if file.split('.')[-1] in filetypes]
    def __get_images_from_directory(self,directory):
        files=[]
        for folder in os.listdir(directory):
            files+=[os.path.join(folder,'comp_pngs',name) for name in os.listdir(os.path.join(directory,folder,'comp_pngs'))]
        files=self.__collect_image_files(files)
        if self.shuffle:
            shuffle(files)
        if self.sample_size !=None:
            files=sample(files,self.sample_size)
        return files


In [7]:
def progress_bar(progress,total,display_length=60):
    left_ratio=display_length*progress//total
    right_ratio=display_length-left_ratio
    print('['+ '='*left_ratio + '>' + '.'*right_ratio + f'] {progress} / {total}', end='\r') 
def plot_reconstruction(image,prediction,num_channels):
    fig,ax=plt.subplots(ncols=3,figsize=(10,3))
    ax[0].title.set_text('Original image')
    ax[0].imshow(image[0])
    ax[1].title.set_text('Reconstructed image')
    ax[1].imshow(prediction[0])
    ax[2].title.set_text('Difference')
    ax[2].imshow(prediction[0]-image[0],cmap='Spectral')
    plt.tight_layout()
    plt.show()
    if num_channels==3:
        fix,ax=plt.subplots(nrows=3,ncols=2,figsize=(10,5))
        ax[0][0].title.set_text('Original Mel Spectogram')
        ax[0][0].imshow(np.array(image[0][:,:,0]), cmap='Reds')
        ax[1][0].title.set_text('Original MFCC')
        ax[1][0].imshow(np.array(image[0][:,:,1]), cmap='Greens')
        ax[2][0].title.set_text('Original Chromagram')
        ax[2][0].imshow(np.array(image[0][:,:,2]), cmap='Blues')
        ax[0][1].title.set_text('Reconstructed Mel Spectogram')
        ax[0][1].imshow(np.array(prediction[0][:,:,0]), cmap='Reds')
        ax[1][1].title.set_text('Reconstructed MFCC')
        ax[1][1].imshow(np.array(prediction[0][:,:,1]), cmap='Greens')
        ax[2][1].title.set_text('Reconstructed Chromagram')
        ax[2][1].imshow(np.array(prediction[0][:,:,2]), cmap='Blues')
        plt.tight_layout()
        plt.show()

In [9]:
img_width=128
img_height=128
kernel_size=5
strides=2

In [11]:
class Time_Freq_Autoencoder_Builder:
    def build(width,height,depth,filters=(32,64,128,256),latent_dim=256,kernel_size=5):
        strides=2
        input_shape=(height,width,depth)
        inputs=Input(shape=input_shape)
        chan_dim=-1
        x_time=Reshape(target_shape=(height,width))(inputs)
        x_freq=Reshape(target_shape=(height,width))(keras.ops.transpose(inputs,axes=[0,1,2,3]))
        for f in filters:
            x_time=Conv1D(f,kernel_size=kernel_size,strides=strides,padding='same',activation='relu')(x_time)
            x_time=BatchNormalization(axis=chan_dim)(x_time)
        x_time=Flatten()(x_time)
        latent_time=Dense(latent_dim//2)(x_time)
        for f in filters:
            x_freq=Conv1D(f,kernel_size=kernel_size,strides=strides,padding='same',activation='relu')(x_freq)
            x_freq=BatchNormalization(axis=chan_dim)(x_freq)
        x_freq=Flatten()(x_freq)
        latent_freq=Dense(latent_dim//2)(x_freq)
        latent_concat=tf.keras.layers.Concatenate(axis=1)([latent_time,latent_freq])
        encoder=Model(inputs,latent_concat,name='encoder')
        latent_inputs=Input(shape=((latent_dim//2)*2,))
        x=Dense(16384,activation='relu')(latent_inputs)
        x=Reshape(target_shape=(8,8,256))(x)
        for f in filters[::-1]:
            x=Conv2DTranspose(f,kernel_size=kernel_size,strides=strides,padding='same',activation='relu')(x)
            x=BatchNormalization(axis=chan_dim)(x)
        x=Conv2DTranspose(depth,kernel_size=kernel_size,padding='same',activation='sigmoid')(x)
        outputs=x
        decoder=Model(latent_inputs,outputs,name='decoder')
        autoencoder=Model(inputs,decoder(encoder(inputs)),name='autoencoder')
        return (encoder,decoder,autoencoder)
class Time_Freq_Autoencoder(tf.keras.Model):
    def __init__(self,image_width,image_height,image_depth=1,latent_dim=256,kernel_size=5,**kwargs):
        #super().__init__(**kwargs)
        super(Time_Freq_Autoencoder, self).__init__(**kwargs)
        self.image_width=image_width
        self.image_height=image_height
        self.image_depth=image_depth
        self.latent_dim=latent_dim
        self.kernel_size=kernel_size
        self.encoder,self.decoder,self.autoencoder=Time_Freq_Autoencoder_Builder.build(width=image_width,height=image_height,depth=image_depth,latent_dim=256,kernel_size=kernel_size)
    def call(self,x):
        autoencoded=self.autoencoder(x)
        return autoencoded
    def get_config(self):
        base_config=super().get_config()
        return{
            **base_config,
            'image_width':self.image_width,
            'image_height':self.image_height,
            'image_depth':self.image_depth,
            'latent_dim':self.latent_dim,
            'kernel_size':self.kernel_size,
        }
    @classmethod
    def from_config(cls,config):
        return cls(**config)
autoencoder=Time_Freq_Autoencoder(image_width=img_width,image_height=img_height,latent_dim=256,kernel_size=5)

In [12]:
opt=Adam(learning_rate=1e-3)
autoencoder.compile(optimizer=opt,loss=tf.keras.losses.mse)

In [13]:
model_path='/kaggle/input/notebooks/hau100416/vae-for-mel-spectrogram/model/autoencoder.keras'
autoencoder=tf.keras.models.load_model(model_path,custom_objects={'Time_Freq_Autoencoder':Time_Freq_Autoencoder})

In [14]:
autoencoder.summary()

Model: "time__freq__autoencoder"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ encoder (Functional)            │ (None, 256)            │     1,000,384 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ decoder (Functional)            │ (None, 128, 128, 1)    │     6,927,489 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ autoencoder (Functional)        │ (None, 128, 128, 1)    │     7,927,873 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,777,861 (90.71 MB)

 Trainable params: 7,924,993 (30.23 MB)

 Non-trainable params: 2,880 (11.25 KB)

 Optimizer params: 15,849,988 (60.46 MB)

In [15]:
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import time
from sklearn.metrics.pairwise import cosine_similarity,euclidean_distances
from sklearn.preprocessing import StandardScaler
from joblib import dump,load
import json
import spotipy 
from spotipy.oauth2 import SpotifyClientCredentials

In [ ]:
class LatentSpace:
    def __init__(self,
                autoencoder_path,
                image_dir,
                track_df_path,sample_size=None,
                latent_dims=128,
                num_channels=1,
                output_size=(128,128),scale=True,threshold_level=0,num_tiles=4):
        self._batch_size=1
        self.autoencoder=tf.keras.models.load_model(autoencoder_path,custom_objects={'Time_Freq_Autoencoder':Time_Freq_Autoencoder})
        self.prediction_generator=AudioDataGenerator(directory=image_dir,
                                                    image_size=(128,512),
                                                    color_mode='rgb',
                                                    batch_size=1,
                                                    shuffle=False,sample_size=sample_size,output_channel_index=0,num_output_channels=num_channels,output_size=output_size,threshold_level=threshold_level)
        self.latent_cols=[f"latent_{i}" for i in range(latent_dims)]
        self._track_df_path=track_df_path
        self.size=self.prediction_generator.size
        self._num_channels=num_channels
        self._scale=scale
        self._num_tiles=num_tiles
        self.clientId=''
        self.clientSecret=''
        credentials_manager=SpotifyClientCredentials(client_id=self.clientId,
                                                    client_secret=self.clientSecret)
        self._spotify=spotipy.Spotify(client_credentials_manager=credentials_manager)
        self._scaler=None
    def build(self):
        results=[]
        print('Getting predictions from autoencoder...')
        start_time=time.time()
        search_range=self.prediction_generator.size
        for i in range(search_range):
            filename,latent_img,_=self.prediction_generator.take(i,return_filename=True,get_all_tiles=True,num_tiles=self._num_tiles)
            latent_space=np.array(self.autoencoder.encoder(latent_img)).mean(axis=0)
            for j in range(self._batch_size):
                result={
                    'id':filename[j].split('.')[0].split('/')[-1],
                    'filename':filename[j],
                }
                for idx,col in enumerate(latent_space):
                    k=f'latent_{idx}'
                    result[k]=col
                results.append(result)
            progress_bar(i+1,search_range)
        print('\n')
        print(round((time.time()-start_time)/60,2),'minutes elapsed')
        start_time=time.time()
        print("Building tracks dataframe")
        results_df=pd.DataFrame(results)
        self.results=results_df
        print('size of results',len(results_df))
        track_df=pd.read_csv(self._track_df_path)
        track_latents=results_df.merge(track_df,how='left',left_on='id',right_on='track_id')
        track_latents=track_latents.drop_duplicates(subset='id')
        track_latents=track_latents.reset_index(drop=True)
        if self._scale:
            self._scaler=StandardScaler()
            track_latent_scaled=self._scaler.fit_transform(track_latents[self.latent_cols])
            track_latents[self.latent_cols]=track_latent_scaled
        self.tracks=track_latents
        print(f'Track dataframe built. {round((time.time()-start_time)/60,2)} minutes elapsed')
        start_time=time.time()
        print("Building artist distributions...")
        artist_latents=track_latents.groupby(['artist_name']).mean(numeric_only=True).reset_index()
        artist_latents.drop(columns=['popularity'],inplace=True)
        self.artists=artist_latents
        print(f'Artist distributions built. {round((time.time()-start_time)/60,2)} minutes elapsed')
        start_time=time.time()
        print('Building genre distributions...')
        genre_rows=[]
        for idx,row in track_latents.iterrows():
                genre=row['genre']
                new_row=row
                new_row['genre']=genre
                genre_rows.append(new_row)
        genre_latents=pd.DataFrame(genre_rows)
        genre_latents=genre_latents.groupby('genre').mean(numeric_only=True).dropna()
        genre_latents=genre_latents.reset_index()
        print(f'Genre distributions built. {round((time.time()-start_time)/60,2)} minutes elapsed')
        self.genres=genre_latents
        print('Latent Space Built.')
    def save(self,directory_to_save,save_full_results=False):
        directories=directory_to_save.split('/')
        save_folders=[directories[0]]
        for directory in directories[1:]:
            save_folders.append(save_folders[-1]+'/'+directory)
        for folder in save_folders:
            try:
                os.mkdir(folder)
            except:
                pass
        self.tracks.to_csv(directory_to_save+'/tracks.csv',index=False)
        self.artists.to_csv(directory_to_save+'/artists.csv',index=False)
        self.genres.to_csv(directory_to_save+'/genres.csv',index=False)
        if self._scale:
            dump(self._scaler,directory_to_save+'/std_scaler.bin',compress=True)
        if save_full_results:
            self.results.to_csv(directory_to_save+'/results.csv',index=False)
    def load(self,directory_to_load,load_full_results=False):
        try:
            self.tracks=pd.read_csv(directory_to_load+'/tracks.csv')
            print('Loaded tracks.')
        except:
            print("Failed to load tracks.")
        try:
            self.artists=pd.read_csv(directory_to_load+'/artists.csv')
            print('Loaded artists.')
        except:
            print('Failed to load artists.')
        try:
            self.genres=pd.read_csv(directory_to_load+'/genres.csv')
            print('Loaded genres.')
        except:
            print('Failed to load genres')
        try:
            self._scaler=load(directory_to_load+'/std_scaler.bin')
            print('loaded scaler')
        except:
            print('Failed to load scaler.')
        if load_full_results:
            try:
                self.genres=pd.read_csv(directory_to_load+"/results.csv")
            except:
                print('Failed to load full results')
            self.prediction_generator.batch_size=1
    def get_similar_tracks_by_index(self, df_index, num=10, similarity_measure='cosine', scope='all'):
        if scope == 'all':
            track_similarity_df = self._get_similarity(self.tracks.iloc[[df_index]], self.tracks, subset=self.latent_cols, num=num, similarity_measure=similarity_measure)
        elif scope == 'similar_artists':
            similar_artist_tracks = pd.DataFrame()
            for artist_id in self.get_similar_artists_by_index(df_index, num=100).artist_id:
                similar_artist_tracks = pd.concat([similar_artist_tracks, self.get_index_by_artist_id(artist_id)])
            similar_artist_tracks = self.tracks[self.tracks.track_id.isin(similar_artist_tracks.track_id)]
            
            track_similarity_df = self._get_similarity(self.tracks.iloc[[df_index]], similar_artist_tracks, subset=self.latent_cols, num=num, similarity_measure=similarity_measure)
        else:
            raise ValueError('scope must be "all" or "similar_artists"')
        
        return track_similarity_df[['index','track_name','artist_name', 'track_uri','similarity']]
    def get_similar_artists_by_index(self, df_index, num=10, similarity_measure='cosine'):

        artist_similarity_df = self._get_similarity(self.tracks.iloc[[df_index]], self.artists, subset=self.latent_cols, num=num, similarity_measure=similarity_measure)

        return artist_similarity_df[['artist_id','artist_name','similarity']]

    def get_similar_genres_by_index(self, df_index, num=10, similarity_measure='cosine'):

        genre_similarity_df = self._get_similarity(self.tracks.iloc[[df_index]], self.genres, subset=self.latent_cols, num=num, similarity_measure=similarity_measure)

        return genre_similarity_df[['genre','similarity']]


    def get_similarity(self, df1, df2, subset, num=10, similarity_measure='cosine', popularity_threshold=0, sort_tracks=True):
        if similarity_measure == 'cosine':
            similarity_measure_fn = cosine_similarity
            sort = False
        elif similarity_measure == 'euclidean':
            similarity_measure_fn = euclidean_distances
            sort = True
        else:
            raise ValueError('similarity_measure must be "cosine" or "euclidean"')

        similarity = similarity_measure_fn(np.array(df1[subset]), np.array(df2[subset]))

        similarity_df = df2.copy()
        similarity_df['similarity'] = similarity.T

        similarity_df = similarity_df[similarity_df.popularity > popularity_threshold]

        if sort_tracks:
            similarity_df = similarity_df.sort_values(by='similarity', ascending=sort).reset_index()

        return similarity_df[:num] 
    def get_image_data_by_index(self, index, get_all_tiles=False, num_tiles=4):
        if index < self.size:
            return self.prediction_generator.take(index, get_all_tiles=get_all_tiles, num_tiles=num_tiles)[0]
        else:
            raise ValueError(f'Index must be less than total size of generator: {self.size}') 
    def get_vector_from_preview_link(self, link, track_id):
        img = self.prediction_generator.get_vector_from_preview_link(link, track_id, num_tiles=self._num_tiles)
        vector = np.array(self.autoencoder.encoder(img[0])).mean(axis=0)
        vector = self._scaler.transform(pd.DataFrame([vector], columns=self.latent_cols))
        vector = pd.DataFrame(vector, columns=self.latent_cols)
        return vector
    
    def search_for_recommendations(self, query, num=10, popularity_threshold=10, get_time_and_freq=False,link=None):
        id_ = self._spotify.search(query, type='track')['tracks']['items'][0]['id']
        track = self._spotify.track(id_)
        print(track['name'])
        print(link)

        if link is not None:

            vector = self.get_vector_from_preview_link(link, id_)
            similarity = self.get_similarity(vector, self.tracks, subset=self.latent_cols, num=num, popularity_threshold=popularity_threshold)
            if get_time_and_freq:
                similarity['time_similarity'] = self.get_similarity(vector, similarity, subset=self.latent_cols[:len(self.latent_cols)//2], num=num, popularity_threshold=popularity_threshold, sort_tracks=False)['similarity']
                similarity['frequency_similarity'] = self.get_similarity(vector, similarity, subset=self.latent_cols[len(self.latent_cols)//2:], num=num, popularity_threshold=popularity_threshold, sort_tracks=False)['similarity']
            
                return similarity[['track_name','artist_name','similarity','popularity','time_similarity','frequency_similarity']]
            else:
                return similarity[['track_name','artist_name','similarity','popularity']]
        else:
            print('No Preview Available. Try a different search.')

In [17]:
latent_space=LatentSpace(autoencoder_path=model_path,
                       image_dir='/kaggle/input/datasets/hau100416/music-mel-spectrogram-dataset/',
                        track_df_path='/kaggle/input/datasets/amitanshjoshi/spotify-1million-tracks/spotify_data.csv',
                       latent_dims=256,
                       output_size=(128,128),
                       num_tiles=64)


Found 211808 files for Generator set


In [ ]:
# latent_space.build()
# latent_space.save('/kaggle/working',save_full_results=True)

In [18]:
latent_space.load('/kaggle/input/notebooks/hau100416/recommender-with-latent-space')

Loaded tracks.
Loaded artists.
Loaded genres.
loaded scaler


In [19]:
latent_space.tracks['track_name'].unique().shape

(8883,)

In [20]:
track_previews=pd.read_csv('/kaggle/input/datasets/hau100416/spotify-sql/track_preview.csv')

In [21]:
track_previews.columns

Index(['track_id', 'preview_link'], dtype='object')

In [24]:
track_preview=track_previews[track_previews['preview_link']!="'0'"]

In [25]:
track_df=pd.read_csv('/kaggle/input/datasets/amitanshjoshi/spotify-1million-tracks/spotify_data.csv')

In [26]:
songs=track_preview.merge(track_df,on=['track_id'],how='inner')

In [29]:
songs.shape

(388550, 21)

In [40]:
prev_links=list(songs[songs['track_name']=='Counting Stars']['preview_link'])
track_ids=list(songs[songs['track_name']=='Counting Stars']['track_id'])

In [42]:
preview_link=prev_links[0]
track_id=track_is[0]

In [43]:
import re
preview_link=re.sub(r"'",'',preview_link)
preview_link.strip()
preview_link

'https://p.scdn.co/mp3-preview/58d2c15d4ca6ceeab2ac0ab56ceea19b7dc17671?cid=17082d21fb854b7a949131af8dd6a181'

In [44]:
vector = latent_space.get_vector_from_preview_link(preview_link, track_id)

In [45]:
vector

,latent_0,latent_1,latent_2,latent_3,latent_4,latent_5,latent_6,latent_7,latent_8,latent_9,...,latent_246,latent_247,latent_248,latent_249,latent_250,latent_251,latent_252,latent_253,latent_254,latent_255
0,0.52684,-0.575755,1.681627,-0.605635,-1.090388,-0.250733,1.631237,-1.243986,-1.80468,0.267055,...,-0.91044,-0.278673,-2.621971,-2.445732,0.617121,2.092854,0.665498,0.710061,2.223314,1.059943


In [46]:
latents = latent_space.get_similarity(vector, latent_space.tracks, subset=latent_space.latent_cols, num=11)

In [48]:
latents = latents[~latents.track_name.apply(lambda x: x.lower()).isin([track_df['track_name'].str.lower()])][:10].reset_index()

In [49]:
latent_space.search_for_recommendations('Counting Stars', num=10, get_time_and_freq=True,link=preview_link)

Counting Stars
https://p.scdn.co/mp3-preview/58d2c15d4ca6ceeab2ac0ab56ceea19b7dc17671?cid=17082d21fb854b7a949131af8dd6a181


,track_name,artist_name,similarity,popularity,time_similarity,frequency_similarity
0,Snowing,Henrik Janson,0.656237,23.0,0.777817,0.545358
1,Hymn #76,Joe Pug,0.639876,24.0,0.689209,0.606528
2,April Sky,Chin Cheng Lin,0.639809,15.0,0.738890,0.568645
3,Theme From MASH,Claes Nilsson,0.607179,14.0,0.688651,0.545538
4,Beat Of The Moment (Remastered),009 Sound System,0.583181,17.0,0.799695,0.443124
5,Fairytale of New York,Manuel Boltano,0.581758,20.0,0.642987,0.540929
6,Pony Blues,Stefan Grossman,0.572247,14.0,0.690239,0.517228
7,La vie en rose,Richard Galliano,0.570512,33.0,0.679047,0.478283
8,Words,Gregory Alan Isakov,0.569362,54.0,0.764943,0.376162
9,Mary Had a Little Lamb and White Noise Ocean W...,Baby Lullaby,0.566124,23.0,0.787410,0.382325
